# Standard Du-DNN Training From New_Dataset.csv

This notebook trains the standard `Du` DNN using `New_Dataset.csv` in `1. Dataset/`, which consolidates all `RandomPushResults*` batches into a single file.

The data process follows the paper-style iterative logic:

1. Load the consolidated CSV and group rows by the `batch` column.
2. Apply the Du > 0 filter for each batch.
3. Split each batch into 80% train and 20% test.
4. Accumulate train/test sets over iterations.
5. Train a fresh standard Du-DNN at each cumulative dataset size.

No project modules are imported; everything needed for this run is defined in this notebook.

In [ ]:
from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Configuration

The hyperparameters below are fixed to the standard Du-DNN values reported in Table 2 of the paper. This notebook focuses on reproducing the data-growth and splitting process, not tuning hyperparameters.

In [ ]:
GLOBAL_SEED = 42
TARGET = 'Du'
DATASET_PATH = Path('../1. Dataset/New_Dataset.csv')

TRAIN_RATIO = 0.80
TEST_RATIO = 0.20
MAX_ITERATIONS = 40
R2_THRESHOLD = 0.90

MAX_BATCH_FOLDERS = 80

INPUT_COLS = [
    'B', 'D', 'H', 'B_num', 'D_num', 'H_num',
    'colWidth', 'beamDepth', 'beamRat', 'Asc', 'Asb',
    't', 'cover1', 'cover2', 'Ec', 'nu_c', 'fc',
    'fcuRat', 'eps_cu', 'Es', 'nu_s', 'fsy', 'eta'
]

OUTPUT_COLS = ['Dy', 'Vy', 'Du', 'Vu', 'IDy', 'IVy', 'IDu', 'IVu']

HYPERPARAMS = {
    'neurons_per_layer': [256, 128, 64, 64, 64, 64],
    'batch_size': 128,
    'weight_decay': 2.0,
    'dropout_rate': 0.30,
    'batch_norm': True,
    'optimizer': 'Adam',
    'initial_lr': 1.0e-5,
    'max_epochs': 3000,
    'early_stopping_patience': 500,
    'loss_function': 'MSE',
}

print(f'Target: {TARGET}')
print(f'Dataset: {DATASET_PATH.resolve()}')
print(f'Per-batch split: {TRAIN_RATIO:.0%} train / {TEST_RATIO:.0%} test')
print(f'Max batch folders: {MAX_BATCH_FOLDERS}')
print(f'Hyperparameters: {HYPERPARAMS}')

## Batch Loading and Processing

Data is loaded from `New_Dataset.csv` (pre-consolidated from all `RandomPushResults*` folders). The CSV contains the 23 input features, 8 converted output columns, and a `batch` column identifying the source folder.

Output conversions already applied in the CSV:

- `Dy` = col0 / 1000 (mm → m)
- `Vy` = col1 (kN)
- `Du` = col2 / 1000 (mm → m)
- `Vu` = col3 (kN)
- `IDy` = col6 × 100 (fraction → percent)
- `IVy` = col7 (kN)
- `IDu` = col8 × 100 (fraction → percent)
- `IVu` = col9 (kN)

For Du-only training, rows are kept when `Du > 0` (all-finite rows were already filtered during CSV creation).

In [ ]:
df_all = pd.read_csv(DATASET_PATH)

def batch_number(name):
    return int(name.replace('RandomPushResults', ''))

batch_names = sorted(df_all['batch'].unique(), key=batch_number)
batch_names = batch_names[:MAX_BATCH_FOLDERS]
print(f'Using {len(batch_names)} batches: {batch_names[0]} through {batch_names[-1]}')

target_idx = OUTPUT_COLS.index(TARGET)

batch_summaries = []
for name in batch_names:
    df_b = df_all[df_all['batch'] == name]
    raw_rows = len(df_b)
    # rows are already finite-filtered in New_Dataset.csv; apply Du > 0 filter here
    kept = df_b[df_b[TARGET] > 0]
    kept_rows = len(kept)
    batch_summaries.append({
        'folder': name,
        'raw_rows': raw_rows,
        'kept_rows': kept_rows,
        'removed_rows': raw_rows - kept_rows,
    })

batch_summary_df = pd.DataFrame(batch_summaries)
display(batch_summary_df.head())
print(
    f"Raw rows used: {batch_summary_df['raw_rows'].sum():,} | "
    f"Kept rows after Du cleaning: {batch_summary_df['kept_rows'].sum():,} | "
    f"Removed rows: {batch_summary_df['removed_rows'].sum():,}"
)

## Split Each Batch 80:20 and Accumulate

Each batch group from the CSV is split once using a fixed seed. The cumulative train/test pools grow as new batches are added:

- iteration 1: train/test split from batch 1
- iteration 2: split from batch 1 + split from batch 2
- iteration 40: split from batches 1 through 40

This keeps the split close to 80:20 at every cumulative iteration while respecting the paper's add-another-batch logic.

In [ ]:
def build_cumulative_splits(names):
    cumulative = []
    train_X_parts, train_y_parts = [], []
    test_X_parts, test_y_parts = [], []

    raw_rows_so_far = 0

    for i, name in enumerate(names, start=1):
        df_b = df_all[df_all['batch'] == name]
        raw_rows_so_far += len(df_b)

        kept = df_b[df_b[TARGET] > 0]
        X_batch = kept[INPUT_COLS].values.astype(np.float32)
        y_batch = kept[TARGET].values.astype(np.float32)

        X_tr, X_te, y_tr, y_te = train_test_split(
            X_batch,
            y_batch,
            train_size=TRAIN_RATIO,
            random_state=GLOBAL_SEED + i,
            shuffle=True,
        )

        train_X_parts.append(X_tr)
        train_y_parts.append(y_tr)
        test_X_parts.append(X_te)
        test_y_parts.append(y_te)

        X_train = np.vstack(train_X_parts).astype(np.float32)
        y_train = np.concatenate(train_y_parts).astype(np.float32)
        X_test = np.vstack(test_X_parts).astype(np.float32)
        y_test = np.concatenate(test_y_parts).astype(np.float32)

        cumulative.append({
            'iteration': i,
            'last_batch': name,
            'X_train_raw': X_train,
            'y_train_raw': y_train,
            'X_test_raw': X_test,
            'y_test_raw': y_test,
            'raw_rows': raw_rows_so_far,
            'kept_rows': int(len(y_train) + len(y_test)),
            'train_rows': int(len(y_train)),
            'test_rows': int(len(y_test)),
        })

    return cumulative


cumulative_splits = build_cumulative_splits(batch_names)

split_summary_df = pd.DataFrame([
    {
        'iteration': s['iteration'],
        'last_batch': s['last_batch'],
        'raw_rows': s['raw_rows'],
        'kept_rows': s['kept_rows'],
        'train_rows': s['train_rows'],
        'test_rows': s['test_rows'],
        'train_ratio': s['train_rows'] / s['kept_rows'],
        'test_ratio': s['test_rows'] / s['kept_rows'],
    }
    for s in cumulative_splits
])

display(split_summary_df.head())
display(split_summary_df.tail())

## Model and Training Utilities

At each iteration, the input scaler is fitted on the cumulative training features only, then applied to the cumulative test features. The target `Du` is kept in physical units of meters because the standard-DNN section of the paper only states that input features were normalized.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class RCFDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y.reshape(-1, 1))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class DNNModel(nn.Module):
    def __init__(self, n_inputs, neurons, dropout_rate, use_batch_norm):
        super().__init__()
        layers = []
        in_dim = n_inputs
        for width in neurons:
            layers.append(nn.Linear(in_dim, width))
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(width))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            in_dim = width
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def make_loaders(X_train, y_train, X_test, y_test, batch_size):
    pin = device.type == 'cuda'
    train_loader = DataLoader(
        RCFDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        pin_memory=pin,
        num_workers=0,
    )
    test_loader = DataLoader(
        RCFDataset(X_test, y_test),
        batch_size=batch_size * 4,
        shuffle=False,
        pin_memory=pin,
        num_workers=0,
    )
    return train_loader, test_loader


def fmt_time(seconds):
    seconds = int(seconds)
    if seconds < 60:
        return f'{seconds}s'
    if seconds < 3600:
        return f'{seconds // 60}m {seconds % 60}s'
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f'{h}h {m}m {s}s'


def print_epoch_log_header():
    print()
    print(
        f"    {'Epoch':>6}  "
        f"{'Train MSE':>12}  "
        f"{'Test MSE':>12}  "
        f"{'Train R2':>9}  "
        f"{'Test R2':>8}  "
        f"{'Time':>12}"
    )
    print(
        f"    {'-' * 6}  "
        f"{'-' * 12}  "
        f"{'-' * 12}  "
        f"{'-' * 9}  "
        f"{'-' * 8}  "
        f"{'-' * 12}"
    )


def print_epoch_log_row(epoch, train_mse, test_mse, train_r2, test_r2, cumulative_time):
    print(
        f"    {epoch:6d}  "
        f"{train_mse:12.6f}  "
        f"{test_mse:12.6f}  "
        f"{train_r2:9.4f}  "
        f"{test_r2:8.4f}  "
        f"{fmt_time(cumulative_time):>12}"
    )

## Train One Cumulative Iteration

In [ ]:
def train_one_cumulative_split(split):
    set_seed(GLOBAL_SEED)

    X_train_raw = split['X_train_raw']
    y_train_raw = split['y_train_raw']
    X_test_raw = split['X_test_raw']
    y_test_raw = split['y_test_raw']

    input_scaler = MinMaxScaler()
    X_train = input_scaler.fit_transform(X_train_raw).astype(np.float32)
    X_test = input_scaler.transform(X_test_raw).astype(np.float32)

    y_train = y_train_raw.astype(np.float32)
    y_test = y_test_raw.astype(np.float32)

    train_loader, test_loader = make_loaders(
        X_train, y_train, X_test, y_test, HYPERPARAMS['batch_size']
    )

    model = DNNModel(
        n_inputs=len(INPUT_COLS),
        neurons=HYPERPARAMS['neurons_per_layer'],
        dropout_rate=HYPERPARAMS['dropout_rate'],
        use_batch_norm=HYPERPARAMS['batch_norm'],
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=HYPERPARAMS['initial_lr'],
        weight_decay=HYPERPARAMS['weight_decay'],
    )

    train_losses = []
    test_losses = []
    epoch_times = []
    best_test_loss = float('inf')
    best_state = None
    no_improve = 0
    train_start = time.perf_counter()

    print_epoch_log_header()

    for epoch in range(HYPERPARAMS['max_epochs']):
        epoch_start = time.perf_counter()

        model.train()
        train_running = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device).squeeze(-1)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            train_running += loss.item() * len(xb)
        train_loss = train_running / len(train_loader.dataset)
        train_losses.append(train_loss)

        model.eval()
        test_running = 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device)
                yb = yb.to(device).squeeze(-1)
                test_running += criterion(model(xb), yb).item() * len(xb)
        test_loss = test_running / len(test_loader.dataset)
        test_losses.append(test_loss)
        epoch_times.append(time.perf_counter() - epoch_start)

        if (epoch + 1) % 500 == 0:
            with torch.no_grad():
                pred_train = model(torch.from_numpy(X_train).to(device)).cpu().numpy()
                pred_test = model(torch.from_numpy(X_test).to(device)).cpu().numpy()
            train_r2 = r2_score(y_train_raw, pred_train)
            test_r2 = r2_score(y_test_raw, pred_test)
            cumulative_time = time.perf_counter() - train_start

            print_epoch_log_row(
                epoch + 1,
                train_loss,
                test_loss,
                train_r2,
                test_r2,
                cumulative_time,
            )

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= HYPERPARAMS['early_stopping_patience']:
                print(f'    Early stopping at epoch {epoch + 1}.')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred_train = model(torch.from_numpy(X_train).to(device)).cpu().numpy()
        pred_test = model(torch.from_numpy(X_test).to(device)).cpu().numpy()

    train_mse_original = mean_squared_error(y_train_raw, pred_train)
    test_mse_original = mean_squared_error(y_test_raw, pred_test)
    train_r2 = r2_score(y_train_raw, pred_train)
    test_r2 = r2_score(y_test_raw, pred_test)

    return {
        'model': model,
        'iteration': split['iteration'],
        'last_batch': split['last_batch'],
        'raw_rows': split['raw_rows'],
        'kept_rows': split['kept_rows'],
        'train_rows': split['train_rows'],
        'test_rows': split['test_rows'],
        'train_losses': train_losses,
        'test_losses': test_losses,
        'train_mse_original': train_mse_original,
        'test_mse_original': test_mse_original,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'epoch_times': epoch_times,
        'input_scaler': input_scaler,
        'y_train_true': y_train_raw,
        'y_test_true': y_test_raw,
        'pred_train': pred_train,
        'pred_test': pred_test,
    }

## Run Iterative Training

In [ ]:
SEP = '=' * 60
print('\n' + SEP)
print(f'  TRAINING FOR OUTPUT: {TARGET}')
print(SEP)

all_results = []
best_result = None
best_test_r2 = -float('inf')

for split in cumulative_splits:
    iteration = split['iteration']
    iter_start = time.perf_counter()

    print(
        f"\n  ITERATION {iteration:3d} | RAW SAMPLES={split['raw_rows']:7,} "
        f"| KEPT={split['kept_rows']:7,} | TRAIN={split['train_rows']:7,} "
        f"| TEST={split['test_rows']:7,}"
    )

    result = train_one_cumulative_split(split)
    iter_elapsed = time.perf_counter() - iter_start
    epoch_total = sum(result['epoch_times'])
    all_results.append(result)

    print(
        f"\n  ITER {iteration:3d} DONE | raw n={result['raw_rows']:7,} | kept n={result['kept_rows']:7,} | "
        f"train={result['train_rows']:7,} | test={result['test_rows']:7,} | "
        f"train MSE={result['train_mse_original']:.6f} | test MSE={result['test_mse_original']:.6f} | "
        f"train R2={result['train_r2']:.4f} | test R2={result['test_r2']:.4f} | "
        f"iter time={fmt_time(iter_elapsed)}  (epochs total={fmt_time(epoch_total)})"
    )

    if result['test_r2'] > best_test_r2:
        best_test_r2 = result['test_r2']
        best_result = result
        print(f"\n  *** New best model with test R2={best_test_r2:.4f}")
    else:
        print(f"\n  --- No improvement (best so far: test R2={best_test_r2:.4f})")

    if result['test_r2'] >= R2_THRESHOLD:
        print(f"\n  >>> R2 threshold {R2_THRESHOLD} reached at raw n={result['raw_rows']:,}. Stopping.")
        break

print(
    f"\n  Best model -- iteration={best_result['iteration']} | raw n={best_result['raw_rows']:,} | "
    f"kept n={best_result['kept_rows']:,} | train={best_result['train_rows']:,} | test={best_result['test_rows']:,} | "
    f"train MSE={best_result['train_mse_original']:.6f}  test MSE={best_result['test_mse_original']:.6f} | "
    f"train R2={best_result['train_r2']:.4f}  test R2={best_result['test_r2']:.4f}"
)

## Results Table

In [ ]:
results_df = pd.DataFrame([
    {
        'iteration': r['iteration'],
        'last_batch': r['last_batch'],
        'raw_rows': r['raw_rows'],
        'kept_rows': r['kept_rows'],
        'train_rows': r['train_rows'],
        'test_rows': r['test_rows'],
        'train_mse': r['train_mse_original'],
        'test_mse': r['test_mse_original'],
        'train_r2': r['train_r2'],
        'test_r2': r['test_r2'],
        'epochs_run': len(r['train_losses']),
        'epoch_time_total': sum(r['epoch_times']),
    }
    for r in all_results
])

display(results_df)

if len(results_df) > 0:
    print('\nBest row:')
    display(results_df.loc[[results_df['test_r2'].idxmax()]])